# Phase 2 — Data Cleaning

## Project
Customer Experience & E-Commerce Performance Investigation Using SQL, Python, Power BI & GenAI

## Objective
Clean and validate the Olist Brazilian E-Commerce Public Dataset while preserving the original raw data.

## Cleaning Principles
- Raw data will never be modified.
- Cleaning decisions will be based on findings from Phase 1.
- Missing values will not be removed automatically.
- Duplicate records will be investigated before removal.
- Referential integrity will be validated after cleaning.
- Date fields will be converted to appropriate datetime types.
- Categorical values will be standardized where appropriate.
- Analytical datasets will be created only after the individual tables are cleaned and validated.

## Phase 1 Findings Carried Forward

The following data-quality and structural issues were identified during Phase 1 and will be investigated during the cleaning process.

### Missing Values
- `orders.order_approved_at` — 160 missing values
- `orders.order_delivered_carrier_date` — 1,783 missing values
- `orders.order_delivered_customer_date` — 2,965 missing values
- `products.product_category_name` — 610 missing values
- Product physical attributes contain a small number of missing values.

### Missing Related Records
- 775 orders are not represented in `order_items`.
- 1 order is not represented in `payments`.
- 768 orders have no review record.

These records will be investigated rather than automatically deleted.

### Duplicate / Key Observations
- `reviews.review_id` contains 814 repeated values.
- The combination `review_id + order_id` is unique.
- `geolocation` contains 261,831 exact duplicate rows.
- `geolocation_zip_code_prefix` is not unique and therefore is not treated as a primary key.
- `order_items` is uniquely identified by `order_id + order_item_id`.
- `payments` is uniquely identified by `order_id + payment_sequential`.

### Category Translation
- 610 products have missing category values.
- 13 product records belong to two categories that do not have English translations.
- The original Portuguese category field will be preserved.

### Geographic Matching
- 278 customer records have ZIP prefixes without a matching geolocation ZIP prefix.
- 7 seller records have ZIP prefixes without a matching geolocation ZIP prefix.
- Geolocation will be treated as a supporting ZIP-prefix lookup rather than a one-to-one reference table.

### Datatype Observations
- Order date columns were initially loaded as `object` and require conversion to datetime.

### Cleaning Principle
The presence of a missing value, duplicate identifier, unmatched reference, or statistical outlier does not automatically mean that a record is invalid.

Each issue will be investigated and a cleaning decision will be made based on the table's grain, relationships, and business meaning.

In [3]:
import pandas as pd
import numpy as np

In [4]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [5]:
tables = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


In [6]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geolocation_clean = geolocation.copy()
category_translation_clean = category_translation.copy()

## 2. Missing Value Analysis

Missing values are not automatically treated as errors. Each missing-value pattern will be investigated based on the table's structure and business meaning before deciding whether to retain, transform, or exclude records.

In [7]:
missing_summary = []

for name, df in tables.items():
    for column in df.columns:
        missing = df[column].isna().sum()
        
        if missing > 0:
            missing_summary.append({
                "table": name,
                "column": column,
                "missing_count": missing,
                "missing_percentage": round(
                    missing / len(df) * 100, 2
                )
            })

missing_summary_df = pd.DataFrame(missing_summary)

missing_summary_df

,table,column,missing_count,missing_percentage
0,orders,order_approved_at,160,0.16
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_delivered_customer_date,2965,2.98
3,reviews,review_comment_title,87656,88.34
4,reviews,review_comment_message,58247,58.70
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01


### Missing Value Findings

The missing-value analysis identified missing data primarily in the Orders, Reviews, and Products tables.

#### Orders
- `order_approved_at`: 160 missing values (0.16%)
- `order_delivered_carrier_date`: 1,783 missing values (1.79%)
- `order_delivered_customer_date`: 2,965 missing values (2.98%)

These delivery-related missing values will be investigated against `order_status` before any records are removed or values are imputed.

#### Reviews
- `review_comment_title`: 87,656 missing values (88.34%)
- `review_comment_message`: 58,247 missing values (58.70%)

Missing review text is not treated as an invalid review because customers can provide a rating without written comments. These values will remain missing in the cleaned review table. Reviews containing text will be used for later NLP and complaint analysis.

#### Products
- `product_category_name`: 610 missing values (1.85%)
- `product_name_lenght`: 610 missing values (1.85%)
- `product_description_lenght`: 610 missing values (1.85%)
- `product_photos_qty`: 610 missing values (1.85%)
- Product physical attributes have only two missing values per column.

The matching count of 610 missing values across the first four product fields will be investigated to determine whether the same product records are affected.

Physical measurements will not be replaced with zero because zero would represent an actual measurement rather than an unknown value.

### Cleaning Decision

No missing values are removed or imputed at this stage. Each missing-value pattern will first be validated against the structure and business meaning of the data.

In [8]:
pd.crosstab(
    orders_clean["order_status"],
    orders_clean["order_delivered_customer_date"].isna()
)

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


### Customer Delivery Date Investigation

The customer delivery date is missing for 2,965 orders.

Most of these missing values occur in orders that are not marked as `delivered`, including `shipped`, `canceled`, `unavailable`, `processing`, `invoiced`, `approved`, and `created` orders. This is generally consistent with the order lifecycle because these orders may not have reached completed customer delivery.

However, 8 orders have `order_status = delivered` while their customer delivery date is missing. These records represent a data-quality anomaly and will be investigated before deciding how to handle them.

Additionally, 6 orders marked as `canceled` contain a customer delivery date. These records will also be investigated rather than automatically removed or modified.

### Decision

No customer delivery dates will be filled or removed at this stage. The anomalous records will first be inspected individually.

In [9]:
delivered_missing_date = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
]

delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [10]:
delivered_missing_details = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
]

delivered_missing_details[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,2018-07-19 00:00:00


In [11]:
delivered_missing_details["order_id"].isin(
    reviews_clean["order_id"]
).value_counts()

order_id
True    8
Name: count, dtype: int64

In [12]:
delivered_missing_details["order_id"].isin(
    order_items_clean["order_id"]
).value_counts()

order_id
True    8
Name: count, dtype: int64

In [13]:
canceled_with_date = orders_clean[
    (orders_clean["order_status"] == "canceled") &
    (orders_clean["order_delivered_customer_date"].notna())
]

canceled_with_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


In [14]:
canceled_with_date[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


### Customer Delivery Date — Investigation Result

The relationship between `order_status` and `order_delivered_customer_date` was investigated.

Most of the 2,965 missing customer delivery dates occur in orders that were not marked as `delivered`, which is structurally understandable because these orders may not have reached completed delivery.

However:
- 8 orders are marked as `delivered` but have no customer delivery date.
- All 8 of these orders have order-item records and review records, confirming that they are legitimate order records with missing delivery timestamps.
- 6 orders are marked as `canceled` but contain customer delivery dates.

### Cleaning Decision

No records were deleted and no order statuses or delivery dates were manually changed.

The 8 delivered orders with missing actual delivery dates will retain their missing values. They will be excluded from calculations that specifically require an actual delivery date.

The 6 canceled orders with delivery dates will also retain their original values. Their `canceled` status will not be changed because the available dataset does not provide sufficient evidence to determine why the final status differs from the fulfillment timestamps.

These anomalies will be documented and considered when defining the population for delivery-performance analysis.

In [15]:
pd.crosstab(
    orders_clean["order_status"],
    orders_clean["order_delivered_carrier_date"].isna()
)

order_delivered_carrier_date,False,True
order_status,,
approved,0,2
canceled,75,550
created,0,5
delivered,96476,2
invoiced,0,314
processing,0,301
shipped,1107,0
unavailable,0,609


In [16]:
delivered_missing_carrier = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_carrier_date"].isna())
]

delivered_missing_carrier

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


In [17]:
delivered_missing_carrier[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


### Carrier Delivery Date — Investigation Result

The carrier delivery date is missing for 1,783 orders.

Most missing values occur in orders that had not reached the carrier stage, such as `created`, `approved`, `processing`, `invoiced`, and `unavailable` orders. This is consistent with the order lifecycle.

There are also 75 canceled orders with a carrier date. These records are retained because the dataset does not provide sufficient evidence to determine whether their final `canceled` status should be changed.

Two orders marked as `delivered` have no carrier delivery date:
- Order `73222` has a customer delivery date but no carrier delivery date.
- Order `92643` has neither a carrier delivery date nor a customer delivery date.

### Cleaning Decision

Missing carrier delivery dates are retained.

No carrier dates are artificially imputed, and no order statuses are changed. The two delivered orders with missing carrier dates will be treated as having unavailable carrier timestamps when performing analyses that require this field.

In [18]:
pd.crosstab(
    orders_clean["order_status"],
    orders_clean["order_approved_at"].isna()
)

order_approved_at,False,True
order_status,,
approved,2,0
canceled,484,141
created,0,5
delivered,96464,14
invoiced,314,0
processing,301,0
shipped,1107,0
unavailable,609,0


In [19]:
delivered_missing_approval = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_approved_at"].isna())
]

delivered_missing_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


In [20]:
delivered_missing_approval[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


In [21]:
canceled_missing_approval = orders_clean[
    (orders_clean["order_status"] == "canceled") &
    (orders_clean["order_approved_at"].isna())
]

pd.crosstab(
    canceled_missing_approval["order_delivered_carrier_date"].isna(),
    canceled_missing_approval["order_delivered_customer_date"].isna()
)

order_delivered_customer_date,True
order_delivered_carrier_date,
True,141


### Orders — Final Missing Date Assessment

The three order lifecycle timestamp fields were investigated against `order_status`.

#### `order_approved_at`
160 values are missing:
- 141 canceled orders also have no carrier or customer delivery dates.
- 5 created orders have no approval timestamp.
- 14 delivered orders have missing approval timestamps but contain later fulfillment timestamps.

#### `order_delivered_carrier_date`
1,783 values are missing:
- Most occur in orders that did not reach the carrier stage.
- 2 delivered orders have missing carrier timestamps.
- 75 canceled orders contain carrier timestamps; these values are retained because the available data does not provide sufficient evidence to change their status.

#### `order_delivered_customer_date`
2,965 values are missing:
- Most occur in orders that did not reach customer delivery.
- 8 delivered orders have missing customer delivery timestamps.
- 6 canceled orders contain customer delivery timestamps; their original status and timestamps are retained.

### Cleaning Decision

No order records were deleted and no missing lifecycle timestamps were artificially imputed.

The original `order_status` values and available timestamps are preserved.

For delivery-performance analysis, metrics requiring actual delivery timestamps will use records where the required timestamp is available and where the order population is appropriate for the specific business question.

The identified anomalies will be documented rather than silently corrected.

In [22]:
pd.crosstab(
    reviews_clean["review_comment_title"].isna(),
    reviews_clean["review_comment_message"].isna()
)

review_comment_message,False,True
review_comment_title,,
False,9839,1729
True,31138,56518


### Reviews — Missing Text Investigation

Review text is optional customer feedback, so missing text values are not automatically considered data-quality errors.

The combination of missing values was investigated across `review_comment_title` and `review_comment_message`:

- 9,839 reviews contain both a title and message.
- 1,729 reviews contain a title but no message.
- 31,138 reviews contain a message but no title.
- 56,518 reviews contain neither title nor message.

Therefore, 42,706 reviews contain at least one textual field, while 56,518 reviews contain no written feedback.

### Cleaning Decision

All 99,224 review records will be retained in `reviews_clean`.

Missing review text will not be imputed or replaced with artificial text.

Reviews without written comments remain useful for numerical satisfaction analysis because `review_score` is available.

A separate text-analysis dataset will later be created for NLP/GenAI analysis using reviews that contain at least one textual field.

In [23]:
products_clean[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().value_counts()

product_category_name  product_name_lenght  product_description_lenght  product_photos_qty
False                  False                False                       False                 32341
True                   True                 True                        True                    610
Name: count, dtype: int64

In [24]:
products_clean[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].isna().value_counts()

product_weight_g  product_length_cm  product_height_cm  product_width_cm
False             False              False              False               32949
True              True               True               True                    2
Name: count, dtype: int64

In [25]:
incomplete_product_ids = products_clean.loc[
    products_clean["product_category_name"].isna(),
    "product_id"
]

incomplete_product_ids.isin(order_items_clean["product_id"]).value_counts()

product_id
True    610
Name: count, dtype: int64

In [26]:
physical_missing_product_ids = products_clean.loc[
    products_clean["product_weight_g"].isna(),
    "product_id"
]

physical_missing_product_ids.isin(order_items_clean["product_id"]).value_counts()

product_id
True    2
Name: count, dtype: int64

### Products — Missing Value Investigation

The missing-value patterns in the Products table were investigated to determine whether the missing fields belonged to the same product records.

#### Catalog Metadata

610 products are missing all four of the following fields simultaneously:

- `product_category_name`
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`

All 610 of these products appear in `order_items`, meaning they have transactional records.

#### Physical Attributes

2 products are missing all four physical attributes simultaneously:

- `product_weight_g`
- `product_length_cm`
- `product_height_cm`
- `product_width_cm`

Both products appear in `order_items`.

### Cleaning Decision

The incomplete product records will be retained.

Missing product metadata and physical attributes will not be artificially imputed because there is insufficient evidence to determine the correct values. In particular, missing physical measurements will not be replaced with zero.

The missing category values will be handled explicitly during later category-level analysis so that transactions involving products with incomplete category information are not silently excluded.

In [27]:
duplicate_summary = []

for name, df in {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean,
    "geolocation": geolocation_clean,
    "category_translation": category_translation_clean
}.items():
    
    duplicate_summary.append({
        "table": name,
        "exact_duplicate_rows": df.duplicated().sum()
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,table,exact_duplicate_rows
0,customers,0
1,orders,0
2,order_items,0
3,payments,0
4,reviews,0
5,products,0
6,sellers,0
7,geolocation,261831
8,category_translation,0


In [28]:
review_id_counts = reviews_clean["review_id"].value_counts()

review_id_counts[review_id_counts > 1].head(20)

review_id
7b606b0d57b078384f0b58eac1d41d78    3
dbdf1ea31790c8ecfcc6750525661a9b    3
32415bbf6e341d5d517080a796f79b5c    3
0c76e7a547a531e7bf9f0b99cba071c1    3
4219a80ab469e3fc9901437b73da3f75    3
abbfacb2964f74f6487c9c10ac46daa6    3
e44840754f12fad2b8646712121b349a    3
70509c441d994fa03d6c1457930c9024    3
2172867fd5b1a55f98fe4608e1547b4b    3
832acec9bbf4efe65c3fb6423d8b4ed7    3
2d6ac45f859465b5c185274a1c929637    3
4d0e6dd087008d1f992d25ef6e1f619f    3
4548534449b1f572e357211b90724f1b    3
f4bb9d6dd4fb6dcc2298f0e7b17b8e1e    3
44e9f871226d8a130de3fc39dfbdf0c5    3
1fb4ddc969e6bea80e38deec00393a6f    3
39b4603793c1c7f5f36d809b4a218664    3
08528f70f579f0c830189efc523d2182    3
38821b5c496b678cf91acc34892805ad    3
9e25d6e3025e9b9a0fc7f03588d33e2b    3
Name: count, dtype: int64

In [30]:
reviews_clean[
    reviews_clean["review_id"] == "7b606b0d57b078384f0b58eac1d41d78"
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7500,7b606b0d57b078384f0b58eac1d41d78,f3028a8f41ea1ee2b461420913663f97,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22
59859,7b606b0d57b078384f0b58eac1d41d78,2deb17060fc1ce18a85eba953ddcdeaf,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22
61069,7b606b0d57b078384f0b58eac1d41d78,2f8f31eb2f7b6572836d662a6625c8e4,5,NaN,NaN,2017-02-15 00:00:00,2017-02-21 23:30:22


### Reviews — Duplicate Identifier Investigation

The Reviews table contains 814 repeated values of `review_id`.

However, investigation of repeated IDs shows that the same `review_id` can be associated with different `order_id` values. For example, the repeated `review_id` `7b606b0d57b078384f0b58eac1d41d78` appears across three different orders.

The composite key `(review_id, order_id)` was previously verified to contain no duplicate combinations.

### Cleaning Decision

`review_id` will not be treated as a unique key by itself.

No review records will be removed based on repeated `review_id` values.

The review records will be retained because there are no exact duplicate rows, and the `(review_id, order_id)` combination uniquely identifies the observed review records.

In [31]:
geolocation_clean[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
5,1012,-23.547762,-46.635361,são paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP


In [32]:
geolocation_clean.drop_duplicates().shape

(738332, 5)

In [33]:
geolocation_clean.shape

(1000163, 5)

In [34]:
geolocation_clean = geolocation_clean.drop_duplicates().copy()

In [35]:
geolocation_clean.shape

(738332, 5)

In [36]:
geolocation_clean.duplicated().sum()

0

### Geolocation — Duplicate Investigation

The Geolocation table contains 1,000,163 rows and does not have a reliable unique identifier.

Investigation showed that a ZIP code prefix can correspond to multiple geographic records with different coordinates, so duplicate ZIP prefixes were not treated as errors.

However, 261,831 rows were found to be exact duplicates across all five columns.

After exact deduplication, the table contains 738,332 rows.

### Cleaning Decision

Only exact duplicate rows were removed from `geolocation_clean`.

Duplicate ZIP code prefixes were retained because the same ZIP prefix can legitimately have multiple geographic records and coordinates.

The Geolocation table continues to be treated as a supporting geographic lookup rather than a one-row-per-ZIP master table.

In [37]:
key_validation = {
    "customers.customer_id": customers_clean["customer_id"].duplicated().sum(),
    "orders.order_id": orders_clean["order_id"].duplicated().sum(),
    "order_items.(order_id, order_item_id)": order_items_clean.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),
    "payments.(order_id, payment_sequential)": payments_clean.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum(),
    "reviews.(review_id, order_id)": reviews_clean.duplicated(
        subset=["review_id", "order_id"]
    ).sum(),
    "products.product_id": products_clean["product_id"].duplicated().sum(),
    "sellers.seller_id": sellers_clean["seller_id"].duplicated().sum(),
    "category_translation.product_category_name": 
        category_translation_clean["product_category_name"].duplicated().sum()
}

key_validation_df = pd.DataFrame(
    list(key_validation.items()),
    columns=["key", "duplicate_count"]
)

key_validation_df

,key,duplicate_count
0,customers.customer_id,0
1,orders.order_id,0
2,"order_items.(order_id, order_item_id)",0
3,"payments.(order_id, payment_sequential)",0
4,"reviews.(review_id, order_id)",0
5,products.product_id,0
6,sellers.seller_id,0
7,category_translation.product_category_name,0


In [38]:
orders_clean[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].dtypes

order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [39]:
order_items_clean["shipping_limit_date"].dtype

dtype('O')

In [42]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews_clean[column] = pd.to_datetime(
        reviews_clean[column],
        errors="coerce"
    )

In [43]:
reviews_clean[review_date_columns].dtypes

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [44]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

orders_clean[order_date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [45]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"],
    errors="coerce"
)

order_items_clean["shipping_limit_date"].dtype

dtype('<M8[ns]')

In [47]:
date_columns_summary = []

for name, df, columns in [
    ("orders", orders_clean, order_date_columns),
    ("order_items", order_items_clean, ["shipping_limit_date"]),
    ("reviews", reviews_clean, review_date_columns)
]:
    for column in columns:
        date_columns_summary.append({
            "table": name,
            "column": column,
            "missing_after_conversion": df[column].isna().sum()
        })

date_columns_summary_df = pd.DataFrame(date_columns_summary)

date_columns_summary_df

,table,column,missing_after_conversion
0,orders,order_purchase_timestamp,0
1,orders,order_approved_at,160
2,orders,order_delivered_carrier_date,1783
3,orders,order_delivered_customer_date,2965
4,orders,order_estimated_delivery_date,0
5,order_items,shipping_limit_date,0
6,reviews,review_creation_date,0
7,reviews,review_answer_timestamp,0


### Datetime Conversion — Validation Result

All relevant date columns were converted from `object` to `datetime64[ns]`.

The missing-value counts were compared after conversion with the counts identified during the initial data-quality assessment. The counts remained unchanged, indicating that no additional missing dates were introduced during conversion.

The original missing lifecycle timestamps in the Orders table were preserved as missing values (`NaT`) rather than being artificially imputed.

In [48]:
for name, df, column in [
    ("customers", customers_clean, "customer_city"),
    ("sellers", sellers_clean, "seller_city"),
    ("geolocation", geolocation_clean, "geolocation_city")
]:
    print(f"\n{name} — {column}")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts().head(10))


customers — customer_city
Unique values: 4119
customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64

sellers — seller_city
Unique values: 611
seller_city
sao paulo         694
curitiba          127
rio de janeiro     96
belo horizonte     68
ribeirao preto     52
guarulhos          50
ibitinga           49
santo andre        45
campinas           41
maringa            40
Name: count, dtype: int64

geolocation — geolocation_city
Unique values: 8011
geolocation_city
sao paulo         79927
rio de janeiro    35177
são paulo         19718
belo horizonte    19474
curitiba          11263
porto alegre       8702
salvador           8083
guarulhos          7411
brasilia           6919
osasco            

In [49]:
import unicodedata

def remove_accents(text):
    if isinstance(text, str):
        return ''.join(
            char for char in unicodedata.normalize("NFD", text)
            if unicodedata.category(char) != "Mn"
        )
    return text

In [50]:
for name, df, column in [
    ("customers", customers_clean, "customer_city"),
    ("sellers", sellers_clean, "seller_city"),
    ("geolocation", geolocation_clean, "geolocation_city")
]:
    standardized = df[column].apply(remove_accents)
    
    print(f"\n{name}")
    print("Original unique:", df[column].nunique())
    print("After accent normalization:", standardized.nunique())
    print("Unique values reduced by:", 
          df[column].nunique() - standardized.nunique())


customers
Original unique: 4119
After accent normalization: 4119
Unique values reduced by: 0

sellers
Original unique: 611
After accent normalization: 610
Unique values reduced by: 1

geolocation
Original unique: 8011
After accent normalization: 5969
Unique values reduced by: 2042


In [53]:
for name, df, column in [
    ("customers", customers_clean, "customer_city"),
    ("sellers", sellers_clean, "seller_city"),
    ("geolocation", geolocation_clean, "geolocation_city")
]:
    normalized = df[column].apply(remove_accents)

    comparison = pd.DataFrame({
        "original": df[column],
        "normalized": normalized
    }).drop_duplicates()

    collisions = comparison[
        comparison.duplicated(subset=["normalized"], keep=False)
    ].sort_values("normalized")

    print(f"\n{name}")
    print(collisions.head(30))


customers
Empty DataFrame
Columns: [original, normalized]
Index: []

sellers
       original normalized
3     sao paulo  sao paulo
360  são paulo  sao paulo

geolocation
            original    normalized
792411     abadiânia     abadiania
792364     abadiania     abadiania
598309        abaete        abaete
598338        abaeté        abaete
697165        abaíra        abaira
697155        abaira        abaira
702417         abaré         abare
702410         abare         abare
888351        abatia        abatia
888733        abatiá        abatia
762201    acailandia    acailandia
762440    açailândia    acailandia
772447         acará         acara
772849         acara         acara
746917        acarau        acarau
746945        acaraú        acarau
754657         acauã         acaua
754676         acaua         acaua
980507        acegua        acegua
980634        aceguá        acegua
779361    acrelândia    acrelandia
778426    acrelandia    acrelandia
808635       acreúna   

In [54]:
def standardize_city(text):
    if isinstance(text, str):
        text = text.strip().lower()
        text = ''.join(
            char for char in unicodedata.normalize("NFD", text)
            if unicodedata.category(char) != "Mn"
        )
        return text
    return text

In [55]:
customers_clean["customer_city_standardized"] = (
    customers_clean["customer_city"].apply(standardize_city)
)

sellers_clean["seller_city_standardized"] = (
    sellers_clean["seller_city"].apply(standardize_city)
)

geolocation_clean["geolocation_city_standardized"] = (
    geolocation_clean["geolocation_city"].apply(standardize_city)
)

In [56]:
print("Customers:")
print(customers_clean["customer_city_standardized"].nunique())

print("\nSellers:")
print(sellers_clean["seller_city_standardized"].nunique())

print("\nGeolocation:")
print(geolocation_clean["geolocation_city_standardized"].nunique())

Customers:
4119

Sellers:
610

Geolocation:
5968


In [57]:
comparison = pd.DataFrame({
    "original": geolocation_clean["geolocation_city"],
    "standardized": geolocation_clean["geolocation_city_standardized"]
}).drop_duplicates()

collisions = comparison[
    comparison.duplicated(subset=["standardized"], keep=False)
].sort_values("standardized")

collisions.head(40)

,original,standardized
792364,abadiania,abadiania
792411,abadiânia,abadiania
598309,abaete,abaete
598338,abaeté,abaete
697165,abaíra,abaira
697155,abaira,abaira
702410,abare,abare
702417,abaré,abare
888351,abatia,abatia
888733,abatiá,abatia


In [58]:
untranslated_categories = products_clean[
    products_clean["product_category_name"].notna() &
    ~products_clean["product_category_name"].isin(
        category_translation_clean["product_category_name"]
    )
]["product_category_name"].value_counts()

untranslated_categories

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

In [59]:
untranslated_product_ids = products_clean.loc[
    products_clean["product_category_name"].isin(
        untranslated_categories.index
    ),
    "product_id"
]

untranslated_product_ids.isin(
    order_items_clean["product_id"]
).value_counts()

product_id
True    13
Name: count, dtype: int64

### Product Categories — Translation Investigation

The product category translation table was compared with the non-null product categories.

13 product records contain valid non-null categories that are not present in the translation table:

- `portateis_cozinha_e_preparadores_de_alimentos` — 10 products
- `pc_gamer` — 3 products

All 13 products appear in `order_items`, confirming that these categories are represented in transactional data.

### Cleaning Decision

The original `product_category_name` values will be retained.

These records will not be removed because the category information itself is present; only the English translation is unavailable in the provided translation table.

When building the analytical dataset, the English category field will be populated where a translation exists, while these untranslated categories will be handled explicitly rather than silently excluded.

In [60]:
for name, df in {
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean
}.items():
    print(f"\n{name}")
    print(df.dtypes)


order_items
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

payments
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

reviews
review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

products
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
produc

### Datatype Validation

Numeric and categorical columns were reviewed after datetime conversion.

Transaction values such as `price`, `freight_value`, and `payment_value` use appropriate numeric types. `review_score` and identifier sequence fields also use appropriate types.

Product metadata fields such as `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` are stored as `float64` because they contain missing values. These fields were not forcibly converted to integer types or imputed solely for datatype consistency.

No additional datatype changes were required at this stage.

In [62]:
order_items_clean[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [63]:
order_items_clean[["price", "freight_value"]].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

,price,freight_value
0.01,9.99,4.4198
0.05,17.00,7.7800
0.25,39.90,13.0800
0.50,74.99,16.2600
0.75,134.90,21.1500
0.95,349.90,45.1200
0.99,890.00,84.5200


In [64]:
Q1 = order_items_clean[["price", "freight_value"]].quantile(0.25)
Q3 = order_items_clean[["price", "freight_value"]].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_summary = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "lower_bound": lower_bound,
    "upper_bound": upper_bound
})

outlier_summary

,Q1,Q3,IQR,lower_bound,upper_bound
price,39.90,134.90,95.00,-102.600,277.400
freight_value,13.08,21.15,8.07,0.975,33.255


In [65]:
for column in ["price", "freight_value"]:
    count = (
        order_items_clean[column] > upper_bound[column]
    ).sum()
    
    print(f"{column}: {count} values above upper IQR bound")

price: 8427 values above upper IQR bound
freight_value: 11613 values above upper IQR bound


In [66]:
order_items_clean[
    [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    "price",
    ascending=False
).head(20)

,order_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01


In [67]:
order_items_clean[
    [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    "freight_value",
    ascending=False
).head(20)

,order_id,product_id,seller_id,price,freight_value
73486,a77e1550db865202c56b19ddc6dc4d53,ec31d2a17b299511e7c8627be9337b9b,257e61d3251fb5efb9daadddbc2cf7ca,979.00,409.68
28044,3fde74c28a3d5d618c00f26d51baafa0,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
3303,076d1555fb53a89b0ef4d529e527a0f6,a3cd9517ebf5a50dca25acce54f3b171,6fa9202c10491e472dffd59a3e82b2a3,2338.08,375.28
69797,9f49bd16053df810384e793386312674,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,1149.00,339.59
16731,264a7e199467906c0727394df82d1a6a,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,338.30
87936,c7a07ddd52bbe18b61da49a8d89853d3,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,322.10
5037,0b6230647ed16f4b3e70282dc4b5b87f,46e24ce614899e36617e37ea1e4aa6ff,17f51e7198701186712e53a39c564617,1050.00,321.88
3584,0822bcde10bb5d023755a71bc8f7797f,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,990.00,321.46
29787,43bdbd9dc0931d72befdf4765af6c442,7e53e051875b2a0c9f22acd8a9a29a20,eeb6de78f79159600292e314a77cbd18,3089.00,317.47
48320,6ddfbf514959b49b6410c01ad93054bb,363a9f5b97bf194da23858be722a7aa5,9596c870880d900012f2e8e6e30d06d7,1045.00,314.40


### Outlier Investigation — Price and Freight

IQR-based analysis identified 8,427 price values and 11,613 freight values above their respective upper IQR bounds.

The IQR method was treated as an outlier-detection technique rather than an automatic data-error rule because both price and freight distributions are strongly right-skewed.

Inspection of extreme transactions showed high-value products and high freight charges that may represent legitimate e-commerce transactions. Some transactions also have freight values substantially higher than product price, which may be analytically relevant to the project's freight-ratio and customer-experience investigation.

### Cleaning Decision

No price or freight records were removed based on statistical outlier detection.

Extreme values will be retained and investigated analytically rather than treated as errors. Later analysis will use appropriate distribution-aware metrics and the derived freight ratio to investigate whether unusually high shipping costs are associated with customer dissatisfaction.

## 3. Referential Integrity Checks

Referential integrity checks verify that values used as foreign keys in one table exist in the corresponding parent table.

The goal is to identify orphan records before creating analytical datasets.

No records will be deleted automatically. Any mismatch will be investigated based on its business meaning.

In [68]:
referential_checks = {
    "orders.customer_id → customers.customer_id":
        orders_clean["customer_id"].isin(customers_clean["customer_id"]).sum(),

    "order_items.order_id → orders.order_id":
        order_items_clean["order_id"].isin(orders_clean["order_id"]).sum(),

    "order_items.product_id → products.product_id":
        order_items_clean["product_id"].isin(products_clean["product_id"]).sum(),

    "order_items.seller_id → sellers.seller_id":
        order_items_clean["seller_id"].isin(sellers_clean["seller_id"]).sum(),

    "payments.order_id → orders.order_id":
        payments_clean["order_id"].isin(orders_clean["order_id"]).sum(),

    "reviews.order_id → orders.order_id":
        reviews_clean["order_id"].isin(orders_clean["order_id"]).sum(),

    "products.category → category_translation.category":
        products_clean["product_category_name"].isin(
            category_translation_clean["product_category_name"]
        ).sum()
}

referential_checks

{'orders.customer_id → customers.customer_id': 99441,
 'order_items.order_id → orders.order_id': 112650,
 'order_items.product_id → products.product_id': 112650,
 'order_items.seller_id → sellers.seller_id': 112650,
 'payments.order_id → orders.order_id': 103886,
 'reviews.order_id → orders.order_id': 99224,
 'products.category → category_translation.category': 32328}

In [69]:
integrity_results = []

checks = [
    ("orders.customer_id → customers.customer_id",
     orders_clean["customer_id"], customers_clean["customer_id"]),

    ("order_items.order_id → orders.order_id",
     order_items_clean["order_id"], orders_clean["order_id"]),

    ("order_items.product_id → products.product_id",
     order_items_clean["product_id"], products_clean["product_id"]),

    ("order_items.seller_id → sellers.seller_id",
     order_items_clean["seller_id"], sellers_clean["seller_id"]),

    ("payments.order_id → orders.order_id",
     payments_clean["order_id"], orders_clean["order_id"]),

    ("reviews.order_id → orders.order_id",
     reviews_clean["order_id"], orders_clean["order_id"]),

    ("products.category → category_translation.category",
     products_clean["product_category_name"].dropna(),
     category_translation_clean["product_category_name"])
]

for relationship, child_values, parent_values in checks:
    unmatched = (~child_values.isin(parent_values)).sum()

    integrity_results.append({
        "relationship": relationship,
        "child_records_checked": len(child_values),
        "unmatched_records": unmatched
    })

integrity_df = pd.DataFrame(integrity_results)

integrity_df

,relationship,child_records_checked,unmatched_records
0,orders.customer_id → customers.customer_id,99441,0
1,order_items.order_id → orders.order_id,112650,0
2,order_items.product_id → products.product_id,112650,0
3,order_items.seller_id → sellers.seller_id,112650,0
4,payments.order_id → orders.order_id,103886,0
5,reviews.order_id → orders.order_id,99224,0
6,products.category → category_translation.category,32341,13


### Referential Integrity — Investigation Result

All major transactional relationships passed referential integrity checks with zero unmatched records.

- Orders → Customers: 0 unmatched
- Order Items → Orders: 0 unmatched
- Order Items → Products: 0 unmatched
- Order Items → Sellers: 0 unmatched
- Payments → Orders: 0 unmatched
- Reviews → Orders: 0 unmatched

The only mismatch was between product categories and the category translation table: 13 non-null product categories were not present in the translation table.

These 13 records belong to two categories:
- `portateis_cozinha_e_preparadores_de_alimentos`: 10 products
- `pc_gamer`: 3 products

These are translation gaps rather than broken product relationships. The affected products exist in the product and order-item tables, so they will be retained. Their original Portuguese category names will be preserved and handled explicitly during analytical preparation.

In [70]:
# Orders without order items
orders_without_items = orders_clean[
    ~orders_clean["order_id"].isin(order_items_clean["order_id"])
]

print("Orders without order items:", len(orders_without_items))

orders_without_items["order_status"].value_counts()

Orders without order items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [71]:
unusual_orders_without_items = orders_without_items[
    ~orders_without_items["order_status"].isin(["unavailable", "canceled"])
]

unusual_orders_without_items

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
7434,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,NaT,NaT,NaT,2018-01-11
9238,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,NaT,NaT,NaT,2018-03-07
21441,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,NaT,NaT,NaT,2017-12-12
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaT,2016-12-01
55086,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,NaT,NaT,NaT,2018-01-08
57591,2ce9683175cdab7d1c95bcbb3e36f478,b2d7ae0415dbbca535b5f7b38056dd1f,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,NaT,NaT,2016-11-25
58958,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,NaT,NaT,NaT,2017-12-01
69926,e04f1da1f48bf2bbffcf57b9824f76e1,0d00d77134cae4c58695086ad8d85100,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,NaT,NaT,2016-11-29


In [72]:
unusual_orders_without_items[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
7434,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,NaT,NaT,NaT,2018-01-11
9238,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,NaT,NaT,NaT,2018-03-07
21441,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,NaT,NaT,NaT,2017-12-12
23254,a68ce1686d536ca72bd2dadc4b8671e5,d7bed5fac093a4136216072abaf599d5,shipped,2016-10-05 01:47:40,2016-10-07 03:11:22,2016-11-07 16:37:37,NaT,2016-12-01
55086,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,NaT,NaT,NaT,2018-01-08
57591,2ce9683175cdab7d1c95bcbb3e36f478,b2d7ae0415dbbca535b5f7b38056dd1f,invoiced,2016-10-05 21:03:33,2016-10-06 07:46:39,NaT,NaT,2016-11-25
58958,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,NaT,NaT,NaT,2017-12-01
69926,e04f1da1f48bf2bbffcf57b9824f76e1,0d00d77134cae4c58695086ad8d85100,invoiced,2016-10-05 13:22:20,2016-10-06 15:51:38,NaT,NaT,2016-11-29


In [73]:
unusual_orders_without_items["order_status"].value_counts()

order_status
created     5
invoiced    2
shipped     1
Name: count, dtype: int64

In [74]:
orders_without_payments = orders_clean[
    ~orders_clean["order_id"].isin(payments_clean["order_id"])
]

print("Orders without payments:", len(orders_without_payments))

orders_without_payments[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

Orders without payments: 1


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


### Orders Without Payment Records — Investigation Result

One order (`30710`) does not have a corresponding payment record.

The order is marked as delivered and contains purchase, approval, carrier-delivery, and customer-delivery timestamps. Therefore, the missing payment record cannot be treated as evidence that the order itself is invalid.

Cleaning decision:
- Retain the order in the cleaned Orders table.
- Do not fabricate a payment record or payment value.
- Do not change the order status.
- Payment-specific analyses will exclude this order where payment information is required.
- Other analyses can continue to use the order's available information.

## 4. Geographic Reference Integrity

Customer and seller ZIP-code prefixes are compared against the geolocation lookup table.

Because the geolocation dataset contains multiple records for the same ZIP prefix, ZIP prefix is treated as a lookup attribute rather than a unique key.

Unmatched ZIP prefixes will be investigated before deciding whether any records should be modified or excluded.

In [76]:
customer_zip_unmatched = customers_clean[
    ~customers_clean["customer_zip_code_prefix"].isin(
        geolocation_clean["geolocation_zip_code_prefix"]
    )
]

print("Unmatched customer ZIP records:", len(customer_zip_unmatched))
print("Unique unmatched customer ZIP prefixes:",
      customer_zip_unmatched["customer_zip_code_prefix"].nunique())

Unmatched customer ZIP records: 278
Unique unmatched customer ZIP prefixes: 157


In [77]:
customer_zip_unmatched["customer_zip_code_prefix"].value_counts().head(20)

customer_zip_code_prefix
70686    15
72005    13
71919    10
73255     7
72300     6
73369     5
72002     5
73401     5
71884     5
72583     4
72595     4
72280     4
71676     4
72596     4
71574     3
29949     3
72017     3
71551     3
72465     3
72821     3
Name: count, dtype: int64

In [78]:
customer_zip_unmatched[
    [
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].sort_values("customer_zip_code_prefix").head(30)

,customer_zip_code_prefix,customer_city,customer_state
60170,2140,sao paulo,SP
3064,6930,cipo-guacu,SP
73328,7412,aruja,SP
32512,7430,aruja,SP
1272,7729,caieiras,SP
79037,7784,cajamar,SP
73365,8342,sao paulo,SP
2166,8980,nossa senhora do remedio,SP
91999,8980,nossa senhora do remedio,SP
382,11547,cubatao,SP


In [79]:
customer_zip_unmatched["customer_city"].value_counts().head(20)

customer_city
brasilia                    171
salvador                      4
sao mateus                    3
maioba                        3
luziania                      3
mage                          2
novo gama                     2
santo eduardo                 2
sao paulo                     2
palmeirinha                   2
maracanau                     2
aracruz                       2
teresina                      2
sinop                         2
monnerat                      2
aruja                         2
nossa senhora do remedio      2
domiciano ribeiro             2
major porto                   1
ipiranga                      1
Name: count, dtype: int64

In [80]:
customer_zip_unmatched["customer_state"].value_counts()

customer_state
DF    171
SP     15
RJ     13
MG     11
PR     11
GO      9
BA      9
ES      6
MA      4
CE      4
RS      4
PE      4
PI      3
PA      3
MT      2
RO      2
PB      2
RN      2
AL      1
SE      1
TO      1
Name: count, dtype: int64

### Customer ZIP Code — Investigation Result

A total of 278 customer records have ZIP-code prefixes that are not present in the geolocation lookup table, representing 157 distinct ZIP prefixes.

The unmatched records contain populated customer city and state fields. The records are distributed across multiple states, with 171 records from DF (Federal District), corresponding to Brasília.

This pattern indicates incomplete coverage in the geolocation lookup rather than sufficient evidence that the customer ZIP codes are invalid.

Cleaning decision:
- Retain all 278 customer records and their original ZIP-code prefixes.
- Do not modify or fabricate ZIP codes or geographic coordinates.
- Treat the geolocation dataset as a supporting lookup with incomplete coverage.
- When geographic enrichment is performed, unmatched ZIP prefixes will remain without geolocation information rather than being artificially populated.

In [81]:
seller_zip_unmatched = sellers_clean[
    ~sellers_clean["seller_zip_code_prefix"].isin(
        geolocation_clean["geolocation_zip_code_prefix"]
    )
]

print("Unmatched seller ZIP records:", len(seller_zip_unmatched))
print("Unique unmatched seller ZIP prefixes:",
      seller_zip_unmatched["seller_zip_code_prefix"].nunique())

Unmatched seller ZIP records: 7
Unique unmatched seller ZIP prefixes: 7


In [82]:
seller_zip_unmatched[
    [
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
]

,seller_zip_code_prefix,seller_city,seller_state
473,82040,curitiba,PR
791,91901,porto alegre,RS
1672,72580,brasilia,DF
1931,2285,sao paulo,SP
2182,7412,aruja,SP
2986,71551,brasilia,DF
3028,37708,pocos de caldas,MG


### Seller ZIP Code — Investigation Result

A total of 7 seller records have ZIP-code prefixes that are not present in the geolocation lookup table.

All 7 unmatched records have distinct ZIP prefixes and contain populated seller city and state fields. The records are distributed across multiple states.

There is insufficient evidence to conclude that these seller ZIP codes are invalid. The mismatch is therefore treated as incomplete coverage in the supporting geolocation lookup.

Cleaning decision:
- Retain all 7 seller records and their original ZIP-code prefixes.
- Do not modify or fabricate ZIP codes or geographic coordinates.
- Treat the geolocation dataset as a supporting lookup with incomplete coverage.
- When geographic enrichment is performed, unmatched seller ZIP prefixes will remain without geolocation information.

## 5. Analytical Dataset Construction

Because the source tables have different levels of granularity, analytical datasets will be constructed at appropriate grains rather than performing unrestricted joins.

Two primary analytical datasets will be created:

1. Order-level dataset — one row per order.
2. Order-item-level dataset — one row per order item.

One-to-many tables such as Payments and Order Items will be aggregated or handled separately before joining to avoid row multiplication and inflated business metrics.

In [83]:
payment_summary = (
    payments_clean
    .groupby("order_id")
    .agg(
        payment_total=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_summary.head()

,order_id,payment_total,payment_count,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [84]:
review_summary = (
    reviews_clean
    .groupby("order_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_score", "count")
    )
    .reset_index()
)

review_summary.head()

,order_id,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [85]:
print("Orders in payment summary:", payment_summary["order_id"].nunique())
print("Orders in review summary:", review_summary["order_id"].nunique())

Orders in payment summary: 99440
Orders in review summary: 98673


In [86]:
order_level = orders_clean.copy()

In [89]:
order_level = order_level.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [90]:
order_level = order_level.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [91]:
print("Order-level shape:", order_level.shape)
print("Unique orders:", order_level["order_id"].nunique())
print("Duplicate order IDs:", order_level["order_id"].duplicated().sum())

Order-level shape: (99441, 18)
Unique orders: 99441
Duplicate order IDs: 0


In [92]:
order_level.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_total_x,payment_count_x,max_installments_x,avg_review_score_x,review_count_x,payment_total_y,payment_count_y,max_installments_y,avg_review_score_y,review_count_y
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,3.0,1.0,4.0,1.0,38.71,3.0,1.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,1.0,1.0,4.0,1.0,141.46,1.0,1.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,1.0,3.0,5.0,1.0,179.12,1.0,3.0,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,1.0,1.0,5.0,1.0,72.20,1.0,1.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,1.0,1.0,5.0,1.0,28.62,1.0,1.0,5.0,1.0


In [93]:
order_level = orders_clean.copy()

print("Starting columns:", order_level.shape[1])

Starting columns: 8


In [94]:
order_level = order_level.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [95]:
print("After payment merge:", order_level.shape)

After payment merge: (99441, 11)


In [96]:
order_level = order_level.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [97]:
print("After review merge:", order_level.shape)

After review merge: (99441, 13)


In [98]:
print("Order-level shape:", order_level.shape)
print("Unique orders:", order_level["order_id"].nunique())
print("Duplicate order IDs:", order_level["order_id"].duplicated().sum())

Order-level shape: (99441, 13)
Unique orders: 99441
Duplicate order IDs: 0


In [99]:
order_level.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_total,payment_count,max_installments,avg_review_score,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,38.71,3.0,1.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,141.46,1.0,1.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,179.12,1.0,3.0,5.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,72.20,1.0,1.0,5.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,28.62,1.0,1.0,5.0,1.0


In [100]:
order_item_level = order_items_clean.copy()

print("Starting shape:", order_item_level.shape)
print("Unique order-item combinations:",
      order_item_level[["order_id", "order_item_id"]].drop_duplicates().shape[0])

Starting shape: (112650, 7)
Unique order-item combinations: 112650


In [101]:
order_item_level = order_item_level.merge(
    products_clean,
    on="product_id",
    how="left"
)

In [102]:
print("After product merge:", order_item_level.shape)
print("Duplicate order-item keys:",
      order_item_level.duplicated(
          subset=["order_id", "order_item_id"]
      ).sum())

After product merge: (112650, 15)
Duplicate order-item keys: 0


In [103]:
order_item_level = order_item_level.merge(
    sellers_clean,
    on="seller_id",
    how="left"
)

In [104]:
print("After seller merge:", order_item_level.shape)

print(
    "Duplicate order-item keys:",
    order_item_level.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

After seller merge: (112650, 19)
Duplicate order-item keys: 0


In [105]:
order_item_level = order_item_level.merge(
    category_translation_clean,
    on="product_category_name",
    how="left"
)

In [106]:
print("After category translation merge:", order_item_level.shape)

print(
    "Duplicate order-item keys:",
    order_item_level.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

After category translation merge: (112650, 20)
Duplicate order-item keys: 0


In [107]:
print(
    "Missing English category:",
    order_item_level["product_category_name_english"].isna().sum()
)

Missing English category: 1627


In [108]:
missing_english_category = order_item_level[
    order_item_level["product_category_name_english"].isna()
]

print("Order-item records with missing English category:",
      len(missing_english_category))

print(
    "Order-item records with missing original category:",
    missing_english_category["product_category_name"].isna().sum()
)

print(
    "Order-item records with non-null but untranslated category:",
    missing_english_category["product_category_name"].notna().sum()
)

Order-item records with missing English category: 1627
Order-item records with missing original category: 1603
Order-item records with non-null but untranslated category: 24


### Category Translation — Analytical Dataset Result

After joining the category translation table, 1,627 order-item records do not have an English category name.

These consist of:
- 1,603 order-item records where the original product category is missing.
- 24 order-item records where the original category exists but is not present in the translation table.

The difference between product-level and order-item-level counts is expected because the analytical dataset contains one row per order item, and a product can appear in multiple orders.

Cleaning decision:
- Preserve the original `product_category_name`.
- Preserve missing values where the original category is unavailable.
- Do not fabricate category names.
- Preserve the original Portuguese categories that lack an English translation.
- Handle missing or untranslated categories explicitly during downstream category analysis.

In [109]:
customer_attributes = orders_clean[
    ["order_id", "customer_id"]
].merge(
    customers_clean[
        [
            "customer_id",
            "customer_city_standardized",
            "customer_state",
            "customer_zip_code_prefix"
        ]
    ],
    on="customer_id",
    how="left"
)

customer_attributes.head()

,order_id,customer_id,customer_city_standardized,customer_state,customer_zip_code_prefix
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,SP,3149
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,BA,47813
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,GO,75265
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,RN,59296
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,SP,9195


In [110]:
print("Customer attribute rows:", len(customer_attributes))
print("Unique orders:", customer_attributes["order_id"].nunique())
print("Duplicate order IDs:",
      customer_attributes["order_id"].duplicated().sum())

Customer attribute rows: 99441
Unique orders: 99441
Duplicate order IDs: 0


In [111]:
order_item_level = order_item_level.merge(
    customer_attributes,
    on="order_id",
    how="left"
)

In [112]:
print("After customer merge:", order_item_level.shape)

print(
    "Duplicate order-item keys:",
    order_item_level.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

After customer merge: (112650, 24)
Duplicate order-item keys: 0


In [113]:
order_level["delivery_delay_days"] = (
    order_level["order_delivered_customer_date"]
    - order_level["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)

In [114]:
order_level[
    [
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].head(10)

,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-10 21:25:13,2017-10-18,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-07 15:27:45,2018-08-13,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-17 18:06:29,2018-09-04,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-12-02 00:28:42,2017-12-15,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-16 18:17:02,2018-02-26,-9.238171
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07-26 10:57:55,2017-08-01,-5.543113
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,NaT,2017-05-09,NaN
7,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05-26 12:55:51,2017-06-07,-11.461215
8,76c6e866289321a7c93b82b54852dc33,delivered,2017-02-02 14:08:10,2017-03-06,-31.410995
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-08-16 17:14:30,2017-08-23,-6.281597


In [115]:
order_level["is_late"] = (
    order_level["delivery_delay_days"] > 0
)

In [116]:
order_level["is_late"].value_counts(dropna=False)

is_late
False    91614
True      7827
Name: count, dtype: int64

In [117]:
order_level["is_late"] = np.where(
    order_level["delivery_delay_days"].isna(),
    np.nan,
    order_level["delivery_delay_days"] > 0
)

In [118]:
order_level["is_late"].value_counts(dropna=False)

is_late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64

In [119]:
order_level["is_late"] = pd.Series(
    np.where(
        order_level["delivery_delay_days"].isna(),
        pd.NA,
        order_level["delivery_delay_days"] > 0
    ),
    dtype="boolean"
)

In [120]:
order_level["is_late"].value_counts(dropna=False)

is_late
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

In [121]:
(
    order_level["order_delivered_customer_date"]
    < order_level["order_purchase_timestamp"]
).sum()

0

### Delivery Timeline — Sanity Check

A timeline validation was performed to identify orders where the recorded customer delivery date occurred before the order purchase timestamp.

Result:
- Orders with delivery before purchase: 0

This indicates no impossible purchase-to-delivery sequence was identified in the cleaned order-level dataset.

No additional correction was required.

In [122]:
order_item_level["freight_ratio"] = (
    order_item_level["freight_value"] /
    order_item_level["price"]
)

In [123]:
order_item_level[
    [
        "order_id",
        "order_item_id",
        "price",
        "freight_value",
        "freight_ratio"
    ]
].head(10)

,order_id,order_item_id,price,freight_value,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,0.225637
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,0.083076
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,0.089799
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,0.090745
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,0.579452
6,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,0.595477
7,000576fe39319847cbb9d288c5617fa6,1,810.00,70.75,0.087346
8,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,0.079822
9,0005f50442cb953dcd1d21e1fb923495,1,53.99,11.40,0.211150


In [124]:
print("Zero-price items:", (order_item_level["price"] == 0).sum())

Zero-price items: 0


In [125]:
order_item_summary = (
    order_item_level
    .groupby("order_id")
    .agg(
        total_product_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count")
    )
    .reset_index()
)

order_item_summary["freight_ratio"] = (
    order_item_summary["total_freight_value"] /
    order_item_summary["total_product_price"]
)

order_item_summary.head()

,order_id,total_product_price,total_freight_value,item_count,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,0.225637
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,0.083076
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,0.089799
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,0.984604
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,0.090745


In [126]:
print("Orders in item summary:", order_item_summary["order_id"].nunique())
print("Duplicate order IDs:",
      order_item_summary["order_id"].duplicated().sum())

Orders in item summary: 98666
Duplicate order IDs: 0


In [127]:
order_level = order_level.merge(
    order_item_summary,
    on="order_id",
    how="left"
)

In [128]:
print("After item summary merge:", order_level.shape)

print(
    "Unique orders:",
    order_level["order_id"].nunique()
)

print(
    "Duplicate order IDs:",
    order_level["order_id"].duplicated().sum()
)

After item summary merge: (99441, 19)
Unique orders: 99441
Duplicate order IDs: 0


In [129]:
order_level[
    order_level["total_product_price"].isna()
][
    [
        "order_id",
        "order_status",
        "total_product_price",
        "total_freight_value",
        "item_count",
        "freight_ratio"
    ]
].head(10)

,order_id,order_status,total_product_price,total_freight_value,item_count,freight_ratio
266,8e24261a7e58791d10cb1bf9da94df5c,unavailable,NaN,NaN,NaN,NaN
586,c272bcd21c287498b4883c7512019702,unavailable,NaN,NaN,NaN,NaN
687,37553832a3a89c9b2db59701c357ca67,unavailable,NaN,NaN,NaN,NaN
737,d57e15fb07fd180f06ab3926b39edcd2,unavailable,NaN,NaN,NaN,NaN
1130,00b1cb0320190ca0daa2c88b35206009,canceled,NaN,NaN,NaN,NaN
1160,2f634e2cebf8c0283e7ef0989f77d217,unavailable,NaN,NaN,NaN,NaN
1579,ee0db22a8e742b752914016708470ec8,unavailable,NaN,NaN,NaN,NaN
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,NaN,NaN,NaN,NaN
1826,6ad57aecbae806a7e9cc2cdb6b380711,unavailable,NaN,NaN,NaN,NaN
1868,df8282afe61008dc26c6c31011474d02,canceled,NaN,NaN,NaN,NaN


In [130]:
print(
    "Orders with missing item metrics:",
    order_level["total_product_price"].isna().sum()
)

Orders with missing item metrics: 775


In [131]:
print("Order-level shape:", order_level.shape)
print("Unique orders:", order_level["order_id"].nunique())
print("Duplicate orders:", order_level["order_id"].duplicated().sum())

print("\nNegative product prices:",
      (order_level["total_product_price"] < 0).sum())

print("Negative freight values:",
      (order_level["total_freight_value"] < 0).sum())

print("Negative item counts:",
      (order_level["item_count"] < 0).sum())

print("Negative freight ratios:",
      (order_level["freight_ratio"] < 0).sum())

print("\nLate flag:")
print(order_level["is_late"].value_counts(dropna=False))

print("\nDelivery delay summary:")
print(order_level["delivery_delay_days"].describe())

Order-level shape: (99441, 19)
Unique orders: 99441
Duplicate orders: 0

Negative product prices: 0
Negative freight values: 0
Negative item counts: 0
Negative freight ratios: 0

Late flag:
is_late
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

Delivery delay summary:
count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64


In [132]:
cleaning_summary = {
    "customers_rows": len(customers_clean),
    "orders_rows": len(orders_clean),
    "order_items_rows": len(order_items_clean),
    "payments_rows": len(payments_clean),
    "reviews_rows": len(reviews_clean),
    "products_rows": len(products_clean),
    "sellers_rows": len(sellers_clean),
    "geolocation_rows": len(geolocation_clean),
    "category_translation_rows": len(category_translation_clean),
    
    "geolocation_duplicates_removed": 261831,
    "orders_without_items": 775,
    "orders_without_payment": 1,
    "customer_zip_unmatched": 278,
    "seller_zip_unmatched": 7,
    "untranslated_categories": 13,
    
    "order_level_rows": len(order_level),
    "order_level_duplicate_orders": order_level["order_id"].duplicated().sum()
}

cleaning_summary

{'customers_rows': 99441,
 'orders_rows': 99441,
 'order_items_rows': 112650,
 'payments_rows': 103886,
 'reviews_rows': 99224,
 'products_rows': 32951,
 'sellers_rows': 3095,
 'geolocation_rows': 738332,
 'category_translation_rows': 71,
 'geolocation_duplicates_removed': 261831,
 'orders_without_items': 775,
 'orders_without_payment': 1,
 'customer_zip_unmatched': 278,
 'seller_zip_unmatched': 7,
 'untranslated_categories': 13,
 'order_level_rows': 99441,
 'order_level_duplicate_orders': 0}

In [133]:
import os

os.makedirs("../data/cleaned", exist_ok=True)
os.makedirs("../data/analytical", exist_ok=True)

In [134]:
customers_clean.to_csv(
    "../data/cleaned/customers_clean.csv",
    index=False
)

orders_clean.to_csv(
    "../data/cleaned/orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    "../data/cleaned/order_items_clean.csv",
    index=False
)

payments_clean.to_csv(
    "../data/cleaned/payments_clean.csv",
    index=False
)

reviews_clean.to_csv(
    "../data/cleaned/reviews_clean.csv",
    index=False
)

products_clean.to_csv(
    "../data/cleaned/products_clean.csv",
    index=False
)

sellers_clean.to_csv(
    "../data/cleaned/sellers_clean.csv",
    index=False
)

geolocation_clean.to_csv(
    "../data/cleaned/geolocation_clean.csv",
    index=False
)

category_translation_clean.to_csv(
    "../data/cleaned/category_translation_clean.csv",
    index=False
)

In [135]:
order_level.to_csv(
    "../data/analytical/order_level.csv",
    index=False
)

order_item_level.to_csv(
    "../data/analytical/order_item_level.csv",
    index=False
)

In [136]:
print("Cleaned and analytical datasets saved successfully.")

Cleaned and analytical datasets saved successfully.


# Phase 2 — Data Cleaning & Analytical Dataset Construction

## Objective

Prepare the Olist Brazilian E-Commerce dataset for reliable business analysis while preserving legitimate business information and avoiding unsupported assumptions or data fabrication.

## Cleaning Principles

- Raw data was preserved and not modified.
- Missing values were investigated based on business meaning rather than blindly removed or imputed.
- Duplicate records were removed only when they were confirmed to be exact duplicates.
- Outliers were investigated but not automatically removed because extreme values may represent legitimate transactions.
- Original categorical values were preserved when standardized versions were created.
- Missing geographic mappings were retained because the geolocation dataset is a supporting lookup and does not cover every ZIP prefix.
- No values were fabricated to fill missing information.
- AI or assumptions were not used to alter the underlying data.

## Data Quality Work Completed

### Missing Values
Investigated missing values across orders, reviews, products and other tables.

Important findings:
- 2,965 orders do not have an actual customer delivery date.
- 1,783 orders do not have a carrier delivery date.
- 160 orders do not have an approval timestamp.
- 56,518 reviews contain no review title or message.
- 610 products have missing category-related information.
- Missing values were retained where they represented legitimate or structurally meaningful cases.

### Duplicates
- Exact duplicate records were checked across all tables.
- Geolocation contained 261,831 exact duplicate rows.
- Exact duplicates were removed from geolocation.
- Other tables contained no exact duplicate rows.
- Reviews had repeated `review_id` values, but `(review_id, order_id)` remained unique, so these records were retained.

### Datetime Standardization
Relevant date columns were converted from object/string format to datetime format using `pd.to_datetime()`.

### City Standardization
Created standardized city columns by:
- trimming whitespace
- converting text to lowercase
- removing accent/diacritic differences

Original city columns were preserved.

### Outlier Investigation
Price and freight values were investigated using IQR-based detection.

Outliers were not automatically removed because extreme transaction values may represent legitimate business activity and could be important for later analysis.

### Referential Integrity
Relationships between major tables were validated using primary-key/foreign-key checks.

No broken relationships were found except 13 non-null product categories without English translations. These were retained as original Portuguese categories.

## Analytical Dataset Construction

### Order-Level Dataset

Created an order-level analytical dataset by combining:

- Orders
- Payment summaries
- Review summaries
- Delivery metrics
- Order-item summaries

Final structure:

- Rows: 99,441
- Unique orders: 99,441
- Duplicate order IDs: 0

Derived metrics include:

- `payment_total`
- `payment_count`
- `max_installments`
- `avg_review_score`
- `review_count`
- `delivery_delay_days`
- `is_late`
- `total_product_price`
- `total_freight_value`
- `item_count`
- `freight_ratio`

### Order-Item-Level Dataset

Created an item-level analytical dataset by combining:

- Order items
- Products
- Sellers
- Category translations
- Customer attributes

The composite key `(order_id, order_item_id)` was validated to remain unique.

## Delivery Metrics

### Delivery Delay

`delivery_delay_days` was calculated as:

Actual customer delivery date − Estimated delivery date

Interpretation:

- Negative = delivered before estimated date
- Zero = delivered on estimated date
- Positive = delivered after estimated date
- Missing = no actual delivery date available

### Late Delivery Flag

`is_late` was created as a nullable Boolean:

- `True` = delivered late
- `False` = delivered on time or early
- `<NA>` = actual delivery date unavailable

Final distribution:

- False: 88,649
- True: 7,827
- <NA>: 2,965

## Important Data Quality Decisions

### Orders Without Items
775 orders have no corresponding order-item records.

These orders were retained rather than assigning artificial revenue or item values.

### Orders Without Payment
1 order has no payment record.

It was retained and will simply be excluded from payment-specific calculations where appropriate.

### Geographic Mismatches
Some customer and seller ZIP prefixes were not found in the geolocation lookup.

These records were retained because missing geolocation data does not imply invalid customer or seller information.

## Final Validation

- Order-level rows = 99,441
- Unique order IDs = 99,441
- Duplicate order IDs = 0
- Negative product prices = 0
- Negative freight values = 0
- Negative item counts = 0
- Negative freight ratios = 0

## Phase 2 Conclusion

The Olist dataset has been cleaned, validated and transformed into analytical datasets suitable for SQL analysis, exploratory data analysis, statistical testing, NLP/review analysis and Power BI.

No unsupported values were fabricated and no legitimate business records were removed solely because they were unusual.

**Phase 2 Status: COMPLETE**